In [ ]:
#Importing necessary libraries:

import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import re
from pygam import LinearGAM, s, f, l
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import numpy as np
import scipy.stats as stats


In [ ]:
#Loading the data into dataframe

df = pd.read_stata("data/UKB 71392 CMD Project.dta")


print(df.shape)

In [ ]:
#Checking datatypes and nulls
df.info()

#Stats of numerical columns
df.describe()

#Number of missing values/column
df.isnull().sum()

list(df.columns)

In [ ]:
#Explorng PRS ADHD data 
df['prs_adhd_std'].describe()

In [ ]:
#Visualising the distribution of PRS ADHD 

plt.figure(figsize=(8, 4))
sns.histplot(df['prs_adhd_std'].dropna(), kde=True)
plt.title('Distribution of Standardized ADHD PRS')
plt.xlabel('ADHD PRS (standardized)')
plt.ylabel('Frequency')
plt.show()

stats.probplot(df['prs_adhd_std'].dropna(), dist="norm", plot=plt)
plt.title('QQ Plot of Standardized ADHD PRS')
plt.show()

In [ ]:
#Cleaning the dataset 

#Fixing the sex column and turning it binary:
df['sex'] = df['sex_plink'].map({1: 0, 2: 1}) #0 => male and 1=> female

#Removing individuals with missing PRS and Key covariates:
main_vars = ['prs_adhd_std', 'n_21001_0_0', 'age_0_0', 'sex',
    'townsend', 'cursmoke_0', 'curr_alcohol_0',
    'PGC1', 'PGC2', 'PGC3', 'PGC4', 'PGC5', 'PGC6', 'PGC7', 'PGC8',
    'chip']

df_model = df.dropna(subset=main_vars).copy()

print(df_model.shape)

In [ ]:
#Visualizing the BMI distribution 

plt.figure(figsize=(10, 6))
sns.histplot(df_model['n_21001_0_0'], kde=True, bins=50, color='skyblue')
plt.title("Distribution of Body Mass Index (BMI)")
plt.xlabel("BMI")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
#Scaling Age and SES factors

scale_cols = ['age_0_0', 'townsend']
scaler = StandardScaler()
df_model[scale_cols] = scaler.fit_transform(df_model[scale_cols])

In [ ]:
#Plotting the BMI distribution into PRS tertiles to visualise trends:

df_model['prs_tertile'] = pd.qcut(df_model['prs_adhd_std'], q =3, labels = ["Low","medium","high"])



# Boxplot of BMI by PRS tertile

plt.figure(figsize=(8, 5))
sns.boxplot(x='prs_tertile', y='n_21001_0_0', data=df_model,  palette='Set2')
plt.title('BMI across ADHD PRS Tertiles')
plt.xlabel('ADHD PRS Tertile')
plt.ylabel('BMI')
plt.show()

# Print mean and median BMI for each PRS tertile
bmi_tertile_summary = (
    df_model
    .groupby('prs_tertile')['n_21001_0_0']
    .agg(['count', 'mean', 'std', 'median', 'min', 'max'])
)

print(bmi_tertile_summary)


In [ ]:
# Model 1: BMI~ ADHD PRS + genetic

#Filter and select relevant columns
cols = ['prs_adhd_std', 'n_21001_0_0', 'chip',
        'PGC1', 'PGC2', 'PGC3', 'PGC4',
        'PGC5', 'PGC6', 'PGC7', 'PGC8']
df_model1 = df_model[cols].dropna().copy()

#Define predictors (X) and outcome (y)
X = df_model1.drop(columns='n_21001_0_0')
X = pd.get_dummies(X, columns=['chip'], drop_first=True, dtype=float)
X = X.apply(pd.to_numeric, errors='coerce')
X = X.dropna()
y = df_model1.loc[X.index, 'n_21001_0_0']

#Add intercept and fit OLS model
X = sm.add_constant(X)
model1 = sm.OLS(y, X).fit()
print(model1.summary())

#Predict and attach predictions back to dataframe
df_model1 = df_model1.loc[X.index]
df_model1['bmi_predicted'] = model1.predict(X)

#Bin the PRS for smoother plotting
df_model1['prs_bin'] = pd.cut(df_model1['prs_adhd_std'], bins=100)
binned = df_model1.groupby('prs_bin')[['prs_adhd_std', 'bmi_predicted']].mean().dropna()

#Plot
plt.figure(figsize=(8, 6))
sns.scatterplot(x='prs_adhd_std', y='n_21001_0_0', data=df_model1,
                alpha=0.1, edgecolor=None, label='Raw Data')

#Smoothed OLS fit
sns.lineplot(x='prs_adhd_std', y='bmi_predicted', data=binned,
        color='red', label='OLS Fit (Smoothed)')

plt.title('Model 1: ADHD PRS + Genetic Covariates vs BMI')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('BMI')
plt.legend()
plt.tight_layout()
plt.show()

print("Model 1")
print("Beta:", df_model1.params["prs_adhd_std"])
print("p-value:", df_model1.pvalues["prs_adhd_std"])


In [ ]:
#Model 2: Model 1+ Age + Sex

#Filter and select relevant columns
cols = ['prs_adhd_std', 'n_21001_0_0', 'chip', 'age_0_0', 'sex_plink',
        'PGC1', 'PGC2', 'PGC3', 'PGC4', 'PGC5', 'PGC6', 'PGC7', 'PGC8']
df_model2 = df_model[cols].dropna().copy()

#Define predictors (X) and outcome (y)
X = df_model2.drop(columns='n_21001_0_0')
X = pd.get_dummies(X, columns=['chip'], drop_first=True, dtype=float)
X = X.apply(pd.to_numeric, errors='coerce')
X = X.dropna()
y = df_model2.loc[X.index, 'n_21001_0_0']

#Add intercept and fit OLS model
X = sm.add_constant(X)
model2 = sm.OLS(y, X).fit()
print(model2.summary())

#Predict and attach predictions back to dataframe
df_model2 = df_model2.loc[X.index]
df_model2['bmi_predicted'] = model2.predict(X)

#Bin the PRS for smoother plotting
df_model2['prs_bin'] = pd.cut(df_model2['prs_adhd_std'], bins=100)
binned = df_model2.groupby('prs_bin')[['prs_adhd_std', 'bmi_predicted']].mean().dropna()

#Plot
plt.figure(figsize=(8, 6))
sns.scatterplot(x='prs_adhd_std', y='n_21001_0_0', data=df_model2,
                alpha=0.1, edgecolor=None, label='Raw Data')

# Smoothed OLS fit
sns.lineplot(x='prs_adhd_std', y='bmi_predicted', data=binned,
                color='red', label='OLS Fit (Smoothed)')
plt.title('Model 2: ADHD PRS + Genetic Covariates + Age + Sex vs BMI')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('BMI')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
#Model 3: Model 2 + lifestyle

#Select relevant columns
cols = ['prs_adhd_std', 'n_21001_0_0', 'age_0_0', 'sex', 'townsend', 
        'cursmoke_0', 'curr_alcohol_0',
        'PGC1', 'PGC2', 'PGC3', 'PGC4', 'PGC5', 'PGC6', 'PGC7', 'PGC8']
df_model3 = df_model[cols].dropna().copy()

#Define predictors and outcome
X3 = df_model3.drop(columns='n_21001_0_0')
X3 = pd.get_dummies(X3, drop_first=True, dtype=float)
X3 = X3.apply(pd.to_numeric, errors='coerce')
X3 = X3.dropna()
y = df_model3.loc[X3.index, 'n_21001_0_0']

#Add intercept and fit the model
X3 = sm.add_constant(X3)
model3 = sm.OLS(y, X3).fit()

#Print summary statistics
print(model3.summary())

#Predict and attach to dataframe
df_model3 = df_model3.loc[X3.index]
df_model3['predicted_bmi_m3'] = model3.predict(X3)

#Bin the PRS values for smoothing
df_model3['prs_bin'] = pd.cut(df_model3['prs_adhd_std'], bins=100)
mean_vals = df_model3.groupby('prs_bin')[['prs_adhd_std', 'predicted_bmi_m3']].mean().dropna()

#Plot
plt.figure(figsize=(10, 7))

#Scatterplot of actual BMI values
sns.scatterplot(x='prs_adhd_std', y='n_21001_0_0', data=df_model3, alpha=0.1, edgecolor=None, label='Observed BMI')

#Lineplot of smoothed predicted BMI
sns.lineplot(x=mean_vals['prs_adhd_std'], y=mean_vals['predicted_bmi_m3'], color='red', label='Model 3 Fit (Smoothed)')

#Labels and formatting
plt.title("Model 3: ADHD PRS + Genetic + Demographic + Lifestyle vs BMI")
plt.xlabel("Standardized ADHD PRS")
plt.ylabel("BMI")
plt.legend()
plt.tight_layout()
plt.show()


This is the end of the analysis on the relationship between PRS ADHD and BMI

In [ ]:
#Analysis of the relationship between PRS ADHD and the outcome of Systolic BP:

#SBP and DBP related columns
sbp_dbp_df = df.filter(regex='sbp|dbp', axis=1)
print("SBP & DBP related columns:\n", sbp_dbp_df.columns.tolist())

#Checking missing values and other stats
summary = pd.DataFrame({
    'Missing Values': sbp_dbp_df.isna().sum(),
    'Data Type': sbp_dbp_df.dtypes})
print(summary)

print(sbp_dbp_df.describe())

#Dropping missing values
df_sbp = df_model.dropna(subset=['adj_sbp_0']).copy()

#Count how many are within -2 to +2
within_2sd = df_sbp[(df_sbp['prs_adhd_std'] >= -2) & (df_sbp['prs_adhd_std'] <= 2)]
proportion_within_2sd = len(within_2sd) / len(df_sbp)

print(f"Proportion of PRS values between -2 and +2: {proportion_within_2sd:.2%}")




In [ ]:
#Visualising the SBP data and trends in it

#Histogram of adjusted SBP distribution
plt.figure(figsize=(10, 6))
sns.histplot(df_sbp['adj_sbp_0'], kde=True, bins=50, color='skyblue')
plt.title("Distribution of Adjusted Systolic Blood Pressure (SBP)")
plt.xlabel("Adjusted SBP")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

#Summary stats

SBP_tertile_summary = (
    df_sbp
    .groupby('prs_adhd_tertile', observed=False)['adj_sbp_0']
    .agg(['count', 'mean', 'std', 'median', 'min', 'max'])
)
print(SBP_tertile_summary)

#Categorize PRS ADHD into tertiles
df_sbp['prs_adhd_tertile'] = pd.qcut(df_sbp['prs_adhd_std'], q=3, labels=['Low', 'Medium', 'High'])

#Plot distribution of SBP across PRS tertiles
plt.figure(figsize=(10, 6))
sns.boxplot(x='prs_adhd_tertile', y='adj_sbp_0', data=df_sbp, palette='Set2')
plt.title("Adjusted SBP Distribution Across ADHD PRS Tertiles")
plt.xlabel("ADHD PRS Tertile")
plt.ylabel("Adjusted SBP")
plt.tight_layout()
plt.show()

In [ ]:
#Model 1: SBP ~ PRS ADHD+ genetic

#Filter and select relevant columns from df_sbp
cols = ['prs_adhd_std', 'adj_sbp_0', 'chip',
        'PGC1', 'PGC2', 'PGC3', 'PGC4',
        'PGC5', 'PGC6', 'PGC7', 'PGC8']
df_sbp1 = df_sbp[cols].dropna().copy()

#Define predictors (X) and outcome (y)
X1_sbp = df_sbp1.drop(columns='adj_sbp_0')
X1_sbp = pd.get_dummies(X1_sbp, columns=['chip'], drop_first=True, dtype=float)
X1_sbp = X1_sbp.apply(pd.to_numeric, errors='coerce')
X1_sbp = X1_sbp.dropna()
y1_sbp = df_sbp1.loc[X1_sbp.index, 'adj_sbp_0']

#Add intercept and fit OLS model
X1_sbp = sm.add_constant(X1_sbp)
model1_sbp = sm.OLS(y1_sbp, X1_sbp).fit()
print(model1_sbp.summary())

#Predict and attach predictions back to dataframe
df_sbp1 = df_sbp1.loc[X1_sbp.index]
df_sbp1['sbp_predicted'] = model1_sbp.predict(X1_sbp)

#Bin PRS for smoothed plotting
df_sbp1['prs_bin'] = pd.cut(df_sbp1['prs_adhd_std'], bins=100)
binned = df_sbp1.groupby('prs_bin')[['prs_adhd_std', 'sbp_predicted']].mean().dropna()

#Plot
plt.figure(figsize=(8, 6))
sns.scatterplot(x='prs_adhd_std', y='adj_sbp_0', data=df_sbp1,
                alpha=0.1, edgecolor=None, label='Raw Data')

#Smoothed OLS fit
sns.lineplot(x='prs_adhd_std', y='sbp_predicted', data=binned,
                color='red', label='OLS Fit (Smoothed)')
plt.title('Model 1: ADHD PRS + Genetic Covariates vs Adjusted SBP')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('Adjusted SBP')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Model 2: SBP ~ Model 1+Age+Sex

#Select relevant columns
cols = ['prs_adhd_std', 'adj_sbp_0', 'age_0_0', 'sex',
        'PGC1', 'PGC2', 'PGC3', 'PGC4',
        'PGC5', 'PGC6', 'PGC7', 'PGC8','chip']
df_sbp2 = df_sbp[cols].dropna().copy()

#Define predictors and outcome
X2 = df_sbp2.drop(columns='adj_sbp_0')
X2 = X2.apply(pd.to_numeric, errors='coerce')
X2 = X2.dropna()
y = df_sbp2.loc[X2.index, 'adj_sbp_0']

#Add intercept and fit model
X2 = sm.add_constant(X2)
model2 = sm.OLS(y, X2).fit()
print(model2.summary())

#Predict values
df_sbp2 = df_sbp2.loc[X2.index]
df_sbp2['predicted_sbp'] = model2.predict(X2)

#Group by ADHD PRS bins for smoothed plot
df_sbp2['prs_bin'] = pd.cut(df_sbp2['prs_adhd_std'], bins=100)
mean_vals = df_sbp2.groupby('prs_bin')[['prs_adhd_std', 'predicted_sbp']].mean().dropna()

#Plot
plt.figure(figsize=(12, 8))
sns.scatterplot(x='prs_adhd_std', y='adj_sbp_0', data=df_sbp2, alpha=0.1, label='Observed SBP')
plt.plot(mean_vals['prs_adhd_std'], mean_vals['predicted_sbp'], color='red', label='Model 2 Fit')
plt.title('Model 2: ADHD PRS + Genetic Covariates + Age + Sex vs Adjusted SBP')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('Adjusted SBP')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
#Model 3: SBP ~ Model 2 + Lifestyle

#Select relevant columns
cols = ['prs_adhd_std', 'adj_sbp_0', 'age_0_0', 'sex', 'townsend',
        'cursmoke_0', 'curr_alcohol_0',
        'PGC1', 'PGC2', 'PGC3', 'PGC4',
        'PGC5', 'PGC6', 'PGC7', 'PGC8','chip']
df_sbp3 = df_sbp[cols].dropna().copy()

#Define predictors and outcome
X3 = df_sbp3.drop(columns='adj_sbp_0')
X3 = X3.apply(pd.to_numeric, errors='coerce')
X3 = X3.dropna()
y = df_sbp3.loc[X3.index, 'adj_sbp_0']

#Add intercept and fit model
X3 = sm.add_constant(X3)
model3 = sm.OLS(y, X3).fit()
print(model3.summary())

#Predict values
df_sbp3 = df_sbp3.loc[X3.index]
df_sbp3['predicted_sbp'] = model3.predict(X3)

#Group by ADHD PRS bins for smoothed plot
df_sbp3['prs_bin'] = pd.cut(df_sbp3['prs_adhd_std'], bins=100)
mean_vals = df_sbp3.groupby('prs_bin')[['prs_adhd_std', 'predicted_sbp']].mean().dropna()

#Plot
plt.figure(figsize=(12, 8))
sns.scatterplot(x='prs_adhd_std', y='adj_sbp_0', data=df_sbp3, alpha=0.1, label='Observed SBP')
plt.plot(mean_vals['prs_adhd_std'], mean_vals['predicted_sbp'], color='red', label='Model 3 Fit')
plt.title('Model 3: ADHD PRS + Genetic + Age + Sex + Lifestyle + SES vs Adjusted SBP')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('Adjusted SBP')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
#Model 4: Model 3 + BMI

#Select relevant columns
cols = ['prs_adhd_std', 'adj_sbp_0', 'age_0_0', 'sex', 'townsend',
        'cursmoke_0', 'curr_alcohol_0', 'n_21001_0_0',  # BMI column
        'PGC1', 'PGC2', 'PGC3', 'PGC4',
        'PGC5', 'PGC6', 'PGC7', 'PGC8','chip']
df_sbp4 = df_sbp[cols].dropna().copy()

#Define predictors and outcome
X4 = df_sbp4.drop(columns='adj_sbp_0')
X4 = X4.rename(columns={'n_21001_0_0': 'bmi'})  # Rename BMI column for clarity
X4 = X4.apply(pd.to_numeric, errors='coerce')
X4 = X4.dropna()
y = df_sbp4.loc[X4.index, 'adj_sbp_0']

#Add intercept and fit model
X4 = sm.add_constant(X4)
model4 = sm.OLS(y, X4).fit()
print(model4.summary())

#Predict values
df_sbp4 = df_sbp4.loc[X4.index]
df_sbp4['predicted_sbp'] = model4.predict(X4)

#Group by ADHD PRS bins for smoothed plot
df_sbp4['prs_bin'] = pd.cut(df_sbp4['prs_adhd_std'], bins=100)
mean_vals = df_sbp4.groupby('prs_bin')[['prs_adhd_std', 'predicted_sbp']].mean().dropna()

#Plot
plt.figure(figsize=(12, 8))
sns.scatterplot(x='prs_adhd_std', y='adj_sbp_0', data=df_sbp4, alpha=0.1, label='Observed SBP')
plt.plot(mean_vals['prs_adhd_std'], mean_vals['predicted_sbp'], color='red', label='Model 4 Fit')
plt.title('Model 4: ADHD PRS + Genetic + Age + Sex + Lifestyle + SES + BMI vs Adjusted SBP')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('Adjusted SBP')
plt.legend()
plt.tight_layout()
plt.show()


SBP is a complex polygenic trait that is a product of several different systems, given the
complexity, genetic influences like the ones captured by ADHD PRS may not be only linear
We use GAMS to explore the relationships more effectively.

In [ ]:
#GAM 1: SBP ~ PRS ADHD+ genetic

#Prepare data
X_gam1 = df_sbp[['prs_adhd_std',
                'PGC1', 'PGC2', 'PGC3', 'PGC4',
                'PGC5', 'PGC6', 'PGC7', 'PGC8',
                'chip']].values
y_gam1 = df_sbp['adj_sbp_0'].values

#Fit model
gam1 = LinearGAM(
    s(0) + s(1) + s(2) + s(3) + s(4) +
    s(5) + s(6) + s(7) + s(8) +
    f(9)
).fit(X_gam1, y_gam1)

#Generate PRS-only grid (term 0), all others set to mean
XX = gam1.generate_X_grid(term=0)

#Predict adjusted SBP and CI
pdep = gam1.predict(XX)
conf_int = gam1.confidence_intervals(XX)  # shape: (100, 2)

#Plot
plt.figure(figsize=(10, 6))
plt.plot(XX[:, 0], pdep, color='red', label='GAM Fit')
plt.fill_between(XX[:, 0], conf_int[:, 0], conf_int[:, 1],
                alpha=0.3, color='blue', label='95% CI')
plt.title("GAM 1: Total Predicted Effect of ADHD PRS on Adjusted SBP")
plt.xlabel("Standardized ADHD PRS")
plt.ylabel("Predicted Adjusted SBP")
plt.legend()
plt.tight_layout()
plt.show()

#Print summary statistics
gam1.summary()

In [ ]:
#GAM 2: SBP ~ GAM 1 + Age + Sex

#Define predictors and response
X_gam2 = df_sbp[['prs_adhd_std', 'age_0_0', 'sex',
                'PGC1', 'PGC2', 'PGC3', 'PGC4', 'PGC5', 'PGC6', 'PGC7', 'PGC8', 'chip']].values
y_gam2 = df_sbp['adj_sbp_0'].values

#Fit the GAM model
gam2 = LinearGAM(
    s(0) +        # ADHD PRS
    s(1) +        # Age
    f(2) +        # Sex
    s(3) + s(4) + s(5) + s(6) + s(7) + s(8) + s(9) + s(10) +  # PGC1–PGC8
    f(11)         # Chip
).fit(X_gam2, y_gam2)

#Building grid for PRS with other variables fixed:
prs_vals = np.linspace(df_sbp['prs_adhd_std'].min(), df_sbp['prs_adhd_std'].max(), 200)
age_mean = df_sbp['age_0_0'].mean()
sex_mode = df_sbp['sex'].mode()[0]
pc_means = df_sbp[['PGC1', 'PGC2', 'PGC3', 'PGC4', 'PGC5', 'PGC6', 'PGC7', 'PGC8']].mean().values
chip_mode = df_sbp['chip'].mode()[0]

# Construct grid with fixed covariates
X_grid = np.column_stack([
    prs_vals,
    np.full_like(prs_vals, age_mean),
    np.full_like(prs_vals, sex_mode),
    np.tile(pc_means, (len(prs_vals), 1)),
    np.full_like(prs_vals, chip_mode)
])

#Get partial dependence and 95% confidence intervals for PRS
pdep, conf_int = gam2.partial_dependence(term=0, X=X_grid, width=0.95)

#Plot
plt.figure(figsize=(10, 6))
plt.plot(prs_vals, pdep, color='red', label='Partial Dependence')
plt.fill_between(prs_vals, conf_int[:, 0], conf_int[:, 1],
                    alpha=0.3, color='blue', label='95% CI')
plt.title("GAM 2: Partial Effect of ADHD PRS on Adjusted SBP\n(Controlling for Age, Sex, and Genetic Covariates)")
plt.xlabel("Standardized ADHD PRS")
plt.ylabel("Partial Effect on SBP")
plt.legend()
plt.tight_layout()
plt.show()

#Summary of model fit
gam2.summary()


In [ ]:
#GAM 3: SBP ~ GAM 2 + Lifestyle

#Define predictors and response
X_gam3 = df_sbp[['prs_adhd_std', 'age_0_0', 'sex',
                'PGC1', 'PGC2', 'PGC3', 'PGC4', 'PGC5', 'PGC6', 'PGC7', 'PGC8', 'chip',
                'townsend', 'cursmoke_0', 'curr_alcohol_0']].values
y_gam3 = df_sbp['adj_sbp_0'].values

# Fit the GAM model
gam3 = LinearGAM(
    s(0) +        # ADHD PRS
    s(1) +        # Age
    f(2) +        # Sex
    s(3) + s(4) + s(5) + s(6) + s(7) + s(8) + s(9) + s(10) +  # PGC1–8
    f(11) +       # Chip
    s(12) +       # Townsend
    f(13) +       # Smoking
    f(14)         # Alcohol
).fit(X_gam3, y_gam3)

# Manually construct grid for ADHD PRS effect
prs_vals = np.linspace(df_sbp['prs_adhd_std'].min(), df_sbp['prs_adhd_std'].max(), 200)
age_mean = df_sbp['age_0_0'].mean()
sex_mode = df_sbp['sex'].mode()[0]
pc_means = df_sbp[['PGC1', 'PGC2', 'PGC3', 'PGC4', 'PGC5', 'PGC6', 'PGC7', 'PGC8']].mean().values
chip_mode = df_sbp['chip'].mode()[0]
townsend_mean = df_sbp['townsend'].mean()
smoke_mode = df_sbp['cursmoke_0'].mode()[0]
alcohol_mode = df_sbp['curr_alcohol_0'].mode()[0]

# Compose full grid
X_grid = np.column_stack([
    prs_vals,
    np.full_like(prs_vals, age_mean),
    np.full_like(prs_vals, sex_mode),
    np.tile(pc_means, (len(prs_vals), 1)),
    np.full_like(prs_vals, chip_mode),
    np.full_like(prs_vals, townsend_mean),
    np.full_like(prs_vals, smoke_mode),
    np.full_like(prs_vals, alcohol_mode)
])

# Partial dependence of PRS (term 0)
pdep, conf_int = gam3.partial_dependence(term=0, X=X_grid, width=0.95)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(prs_vals, pdep, color='red', label='Partial Dependence')
plt.fill_between(prs_vals, conf_int[:, 0], conf_int[:, 1],
                    alpha=0.3, color='blue', label='95% CI')
plt.title("GAM 3: Partial Effect of ADHD PRS on Adjusted SBP\n(Controlling for Age, Sex, Genetics, and Lifestyle)")
plt.xlabel("Standardized ADHD PRS")
plt.ylabel("Partial Effect on SBP")
plt.legend()
plt.tight_layout()
plt.show()

# Model summary
gam3.summary()


In [ ]:
#GAM 4: SBP ~ GAM 3 + BMI

#Define predictors and response
X_gam4 = df_sbp[['prs_adhd_std', 'age_0_0', 'sex', 'townsend', 
                'cursmoke_0', 'curr_alcohol_0',
                'PGC1', 'PGC2', 'PGC3', 'PGC4', 'PGC5', 'PGC6', 'PGC7', 'PGC8', 'chip',
                'n_21001_0_0']].values
y_gam4 = df_sbp['adj_sbp_0'].values

#Fit the GAM model
gam4 = LinearGAM(
    s(0) +   # ADHD PRS
    s(1) +   # Age
    f(2) +   # Sex
    s(3) +   # Townsend
    f(4) +   # Smoking
    f(5) +   # Alcohol
    s(6) + s(7) + s(8) + s(9) + s(10) + s(11) + s(12) + s(13) +  # PGC1–PGC8
    f(14) +  # Chip
    s(15)    # BMI
).fit(X_gam4, y_gam4)

#Step: Build a custom grid over PRS ADHD with all other variables fixed to mean/mode
prs_vals = np.linspace(df_sbp['prs_adhd_std'].min(), df_sbp['prs_adhd_std'].max(), 200)
age_mean = df_sbp['age_0_0'].mean()
sex_mode = df_sbp['sex'].mode()[0]
townsend_mean = df_sbp['townsend'].mean()
smoke_mode = df_sbp['cursmoke_0'].mode()[0]
alcohol_mode = df_sbp['curr_alcohol_0'].mode()[0]
bmi_mean = df_sbp['n_21001_0_0'].mean()
pc_means = df_sbp[['PGC1','PGC2','PGC3','PGC4','PGC5','PGC6','PGC7','PGC8']].mean().values
chip_mode = df_sbp['chip'].mode()[0]

#Compose full grid
X_grid = np.column_stack([
    prs_vals,
    np.full_like(prs_vals, age_mean),
    np.full_like(prs_vals, sex_mode),
    np.full_like(prs_vals, townsend_mean),
    np.full_like(prs_vals, smoke_mode),
    np.full_like(prs_vals, alcohol_mode),
    np.tile(pc_means, (len(prs_vals), 1)),
    np.full_like(prs_vals, chip_mode),
    np.full_like(prs_vals, bmi_mean)
])

#Partial dependence for PRS (term 0)
pdep, confi = gam4.partial_dependence(term=0, X=X_grid, width=0.95)

#Plot
plt.figure(figsize=(10, 6))
plt.plot(prs_vals, pdep, color='red', label='Partial Dependence')
plt.fill_between(prs_vals, confi[:, 0], confi[:, 1],
                alpha=0.3, color='blue', label='95% CI')
plt.title("GAM 4: Partial Effect of ADHD PRS on Adjusted SBP\n(Controlling for Age, Sex, Lifestyle, BMI, Genetic Covariates)")
plt.xlabel("Standardized ADHD PRS")
plt.ylabel("Partial Effect on SBP")
plt.legend()
plt.tight_layout()
plt.show()

#View model summary
gam4.summary()


This is the end of the analysis of a non linear relationship between PRS_ADHD and SBP

In [ ]:
#Analysis of the relationship between DBP and PRS_ADHD

#Filter DBP-related columns
dbp_df = df.filter(regex='dbp', axis=1)
print("DBP-related columns:\n", dbp_df.columns.tolist())

#Check for missing values and data types
dbp_summary = pd.DataFrame({
    'Missing Values': dbp_df.isna().sum(),
    'Data Type': dbp_df.dtypes
})
print(dbp_summary)

#Summary statistics
print(dbp_df.describe())

#Drop missing values for adj_dbp_0
df_dbp = df.dropna(subset=['adj_dbp_0']).copy()

#Count how many PRS_ADHD_STD values are within -2 to +2 SD
within_2sd_dbp = df_dbp[(df_dbp['prs_adhd_std'] >= -2) & (df_dbp['prs_adhd_std'] <= 2)]
proportion_within_2sd_dbp = len(within_2sd_dbp) / len(df_dbp)

print(f"Proportion of PRS_ADHD_STD values between -2 and +2: {proportion_within_2sd_dbp:.2%}")

In [ ]:
#Histogram of adjusted DBP distribution
plt.figure(figsize=(10, 6))
sns.histplot(df_dbp['adj_dbp_0'], kde=True, bins=50, color='skyblue')
plt.title("Distribution of Adjusted Diastolic Blood Pressure (DBP)")
plt.xlabel("Adjusted DBP")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

#Categorize PRS_ADHD_STD into tertiles
df_dbp['prs_adhd_tertile'] = pd.qcut(df_dbp['prs_adhd_std'], q=3, labels=['Low', 'Medium', 'High'])

#Summary stats

dbp_tertile_summary = (
    df_dbp
    .groupby('prs_adhd_tertile', observed=False)['adj_dbp_0']
    .agg(['count', 'mean', 'std', 'median', 'min', 'max'])
)
print(dbp_tertile_summary)


#Boxplot of Adjusted DBP across PRS ADHD tertiles
plt.figure(figsize=(10, 6))
sns.boxplot(x='prs_adhd_tertile', y='adj_dbp_0', data=df_dbp, palette='Set2')
plt.title("Adjusted DBP Distribution Across ADHD PRS Tertiles")
plt.xlabel("ADHD PRS Tertile")
plt.ylabel("Adjusted DBP")
plt.tight_layout()
plt.show()


In [ ]:
#Model 1: DBP ~ PRS ADHD + genetic 

#Filter and select relevant columns from df_dbp (or df if DBP is stored in the same df_sbp)
cols = ['prs_adhd_std', 'adj_dbp_0', 'chip',
        'PGC1', 'PGC2', 'PGC3', 'PGC4',
        'PGC5', 'PGC6', 'PGC7', 'PGC8']
df_dbp1 = df[cols].dropna().copy()

#Define predictors (X) and outcome (y)
X1_dbp = df_dbp1.drop(columns='adj_dbp_0')
X1_dbp = pd.get_dummies(X1_dbp, columns=['chip'], drop_first=True, dtype=float)
X1_dbp = X1_dbp.apply(pd.to_numeric, errors='coerce')
X1_dbp = X1_dbp.dropna()
y1_dbp = df_dbp1.loc[X1_dbp.index, 'adj_dbp_0']

#Add intercept and fit OLS model
X1_dbp = sm.add_constant(X1_dbp)
model1_dbp = sm.OLS(y1_dbp, X1_dbp).fit()
print(model1_dbp.summary())

#Predict and attach predictions back to dataframe
df_dbp1 = df_dbp1.loc[X1_dbp.index]
df_dbp1['dbp_predicted'] = model1_dbp.predict(X1_dbp)

#Bin PRS for smoothed plotting
df_dbp1['prs_bin'] = pd.cut(df_dbp1['prs_adhd_std'], bins=100)
binned = df_dbp1.groupby('prs_bin')[['prs_adhd_std', 'dbp_predicted']].mean().dropna()

#Plot
plt.figure(figsize=(8, 6))
sns.scatterplot(x='prs_adhd_std', y='adj_dbp_0', data=df_dbp1,
                alpha=0.1, edgecolor=None, label='Raw Data')

#Smoothed OLS fit
sns.lineplot(x='prs_adhd_std', y='dbp_predicted', data=binned,
                color='red', label='OLS Fit (Smoothed)')

plt.title('Model 1: ADHD PRS + Genetic Covariates vs Adjusted DBP')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('Adjusted DBP')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
#Model 2: DBP ~ Model 1 + Age +Sex 

#Drop rows with missing values in required columns
cols = ['adj_dbp_0', 'prs_adhd_std', 'age_0_0', 'sex_plink',
        'PGC1', 'PGC2', 'PGC3', 'PGC4',
        'PGC5', 'PGC6', 'PGC7', 'PGC8', 'chip']
df_model2 = df[cols].dropna().copy()

#Define predictors and outcome
X2 = df_model2.drop(columns='adj_dbp_0')
X2 = pd.get_dummies(X2, columns=['chip'], drop_first=True, dtype=float)
X2 = X2.apply(pd.to_numeric, errors='coerce')
X2 = X2.dropna()
y2 = df_model2.loc[X2.index, 'adj_dbp_0']

#Add intercept and fit OLS model
X2 = sm.add_constant(X2)
model2 = sm.OLS(y2, X2).fit()
print(model2.summary())

#Predict DBP
df_model2 = df_model2.loc[X2.index]
df_model2['dbp_predicted'] = model2.predict(X2)

#Bin PRS for smoothed plotting
df_model2['prs_bin'] = pd.cut(df_model2['prs_adhd_std'], bins=100)
binned = df_model2.groupby('prs_bin')[['prs_adhd_std', 'dbp_predicted']].mean().dropna()

#Plot
plt.figure(figsize=(10, 7))
sns.scatterplot(x='prs_adhd_std', y='adj_dbp_0', data=df_model2,
                alpha=0.1, edgecolor=None, label='Raw Data')

#Smoothed OLS fit
sns.lineplot(x='prs_adhd_std', y='dbp_predicted', data=binned,
                color='red', label='OLS Fit (Smoothed)')
plt.title('Model 2: ADHD PRS + Age + Sex + Genetic Covariates vs Adjusted DBP')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('Adjusted DBP')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
#Model 3: DBP~ Model 2 + Lifestyle

#Drop rows with missing values in all required columns
cols = ['adj_dbp_0', 'prs_adhd_std', 'age_0_0', 'sex_plink',
        'townsend', 'cursmoke_0', 'curr_alcohol_0',
        'PGC1', 'PGC2', 'PGC3', 'PGC4',
        'PGC5', 'PGC6', 'PGC7', 'PGC8', 'chip']
df_model3 = df[cols].dropna().copy()

#Define predictors (X) and outcome (y)
X3 = df_model3.drop(columns='adj_dbp_0')
X3 = pd.get_dummies(X3, columns=['chip'], drop_first=True, dtype=float)
X3 = X3.apply(pd.to_numeric, errors='coerce')
X3 = X3.dropna()
y3 = df_model3.loc[X3.index, 'adj_dbp_0']

#Add intercept and fit OLS model
X3 = sm.add_constant(X3)
model3 = sm.OLS(y3, X3).fit()
print(model3.summary())

#Predict DBP
df_model3 = df_model3.loc[X3.index]
df_model3['dbp_predicted'] = model3.predict(X3)

#Bin PRS for smoothed plotting
df_model3['prs_bin'] = pd.cut(df_model3['prs_adhd_std'], bins=100)
binned = df_model3.groupby('prs_bin')[['prs_adhd_std', 'dbp_predicted']].mean().dropna()

#Plot
plt.figure(figsize=(10, 7))
sns.scatterplot(x='prs_adhd_std', y='adj_dbp_0', data=df_model3,
                alpha=0.1, edgecolor=None, label='Raw Data')

#Smoothed OLS fit
sns.lineplot(x='prs_adhd_std', y='dbp_predicted', data=binned,
                color='red', label='OLS Fit (Smoothed)')
plt.title('Model 3: ADHD PRS + SES + Lifestyle + Genetic Covariates vs Adjusted DBP')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('Adjusted DBP')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
#Model 4: DBP ~ Model 3 + BMI

#List of genetic covariates
genetic_covariates = ['PGC1', 'PGC2', 'PGC3', 'PGC4', 
                    'PGC5', 'PGC6', 'PGC7', 'PGC8', 'chip']

#Drop rows with missing values across all needed predictors
df_model4 = df.dropna(subset=[
    'adj_dbp_0', 'prs_adhd_std', 'age_0_0', 'sex_plink', 'townsend',
    'cursmoke_0', 'curr_alcohol_0', 'n_21001_0_0'
] + genetic_covariates).copy()

#Define predictors
X4 = df_model4[['prs_adhd_std', 'age_0_0', 'sex_plink', 'townsend',
                'cursmoke_0', 'curr_alcohol_0', 'n_21001_0_0'] + genetic_covariates]

#One-hot encode 'chip'
X4 = pd.get_dummies(X4, columns=['chip'], drop_first=True, dtype=float)
X4 = X4.apply(pd.to_numeric, errors='coerce').dropna()

#Define outcome
y4 = df_model4.loc[X4.index, 'adj_dbp_0']

#Fit OLS model
X4 = sm.add_constant(X4)
model4 = sm.OLS(y4, X4).fit()
print(model4.summary())

#Predict and prepare for smooth plot
df_model4 = df_model4.loc[X4.index].copy()
df_model4['predicted_dbp_m4'] = model4.predict(X4)

#Bin PRS for smooth plotting
df_model4['prs_bin'] = pd.cut(df_model4['prs_adhd_std'], bins=100)
binned = df_model4.groupby('prs_bin')[['prs_adhd_std', 'predicted_dbp_m4']].mean().dropna()

#Plot
plt.figure(figsize=(10, 7))
sns.scatterplot(x='prs_adhd_std', y='adj_dbp_0', data=df_model4,
                alpha=0.1, edgecolor=None, label='Observed DBP')
sns.lineplot(x='prs_adhd_std', y='predicted_dbp_m4', data=binned,
                color='red', label='Model 4 Fit (Smoothed)')
plt.title("Model 4: ADHD PRS + Lifestyle + BMI + Genetic Covariates vs Adjusted DBP")
plt.xlabel("Standardized ADHD PRS")
plt.ylabel("Adjusted DBP")
plt.legend()
plt.tight_layout()
plt.show()


End of the linear analysis between PRS_adhd_std and DBP

In [ ]:
#GAM 1: DBP ~ PRS ADHD + genetic 

#Columns required
columns_needed = ['adj_dbp_0', 'prs_adhd_std',
                    'PGC1', 'PGC2', 'PGC3', 'PGC4',
                    'PGC5', 'PGC6', 'PGC7', 'PGC8', 'chip']

#Drop rows with any missing values in predictors or outcome
df_gam1_dbp = df[columns_needed].dropna().copy()

#Define X and y
X_gam1_dbp = df_gam1_dbp.drop(columns='adj_dbp_0').values
y_gam1_dbp = df_gam1_dbp['adj_dbp_0'].values

#Fit the GAM
gam1_dbp = LinearGAM(
    s(0) + s(1) + s(2) + s(3) + s(4) +
    s(5) + s(6) + s(7) + s(8) +
    f(9)
).fit(X_gam1_dbp, y_gam1_dbp)

#Generate grid for PRS only (term 0)
XX_dbp = gam1_dbp.generate_X_grid(term=0)

#Predict and CI
pdep_dbp = gam1_dbp.predict(XX_dbp)
conf_int_dbp = gam1_dbp.confidence_intervals(XX_dbp)

#Plot
plt.figure(figsize=(10, 6))
plt.plot(XX_dbp[:, 0], pdep_dbp, color='red', label='GAM Fit')
plt.fill_between(XX_dbp[:, 0], conf_int_dbp[:, 0], conf_int_dbp[:, 1],
                    alpha=0.3, color='blue', label='95% CI')

plt.title("GAM 1: Total Predicted Effect of ADHD PRS on Adjusted DBP")
plt.xlabel("Standardized ADHD PRS")
plt.ylabel("Predicted Adjusted DBP")
plt.legend()
plt.tight_layout()
plt.show()

# Summary
gam1_dbp.summary()

In [ ]:
#GAM 2: DBP ~ GAM 1 + Age + Sex

#Prepare data
genetic_covariates = ['PGC1', 'PGC2', 'PGC3', 'PGC4', 'PGC5', 'PGC6', 'PGC7', 'PGC8', 'chip']
cols_needed = ['prs_adhd_std', 'age_0_0', 'sex_plink', 'adj_dbp_0'] + genetic_covariates

df_gam2 = df.dropna(subset=cols_needed).copy()

#Recode categorical variables
df_gam2['sex_model'] = df_gam2['sex_plink'] + 2
df_gam2['chip_model'] = df_gam2['chip'].astype('category').cat.codes + 1

#Build input matrix and target
X_gam2 = df_gam2[[
    'prs_adhd_std', 'age_0_0', 'sex_model',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8', 'chip_model'
]].values

y_gam2 = df_gam2['adj_dbp_0'].values

#Fit model
gam2 = LinearGAM(
    s(0) + s(1) + f(2) +  # PRS, age, sex
    s(3) + s(4) + s(5) + s(6) + s(7) + s(8) + s(9) + s(10) +  # PGC1–PGC8
    f(11)  # chip
).fit(X_gam2, y_gam2)

#Build grid for PRS effect only (term 0)
prs_vals = np.linspace(df_gam2['prs_adhd_std'].min(), df_gam2['prs_adhd_std'].max(), 200)
age_mean = df_gam2['age_0_0'].mean()
sex_mode = df_gam2['sex_model'].mode()[0]
chip_mode = df_gam2['chip_model'].mode()[0]
pgc_means = [df_gam2[pgc].mean() for pgc in ['PGC1', 'PGC2', 'PGC3', 'PGC4', 'PGC5', 'PGC6', 'PGC7', 'PGC8']]

X_grid = np.column_stack([
    prs_vals,
    np.full_like(prs_vals, age_mean),
    np.full_like(prs_vals, sex_mode),
    *[np.full_like(prs_vals, mean) for mean in pgc_means],
    np.full_like(prs_vals, chip_mode)
])

#Predict partial dependence and confidence interval
pdep, confi = gam2.partial_dependence(term=0, X=X_grid, width=0.95)

#Plot
plt.figure(figsize=(10, 6))
plt.plot(prs_vals, pdep, color='red', label='GAM Fit')
plt.fill_between(prs_vals, confi[:, 0], confi[:, 1], alpha=0.3, color='blue', label='95% CI')
plt.title("GAM 2: Effect of ADHD PRS on Adjusted DBP\n(Adjusted for Age, Sex, and Genetic Covariates)")
plt.xlabel("Standardized ADHD PRS")
plt.ylabel("Partial Effect on DBP")
plt.legend()
plt.tight_layout()
plt.show()

#Summary
gam2.summary()


In [ ]:
#GAM 3: DBP ~ GAM 2 + lifsetyle

#Clean data
genetic_covariates = ['PGC1', 'PGC2', 'PGC3', 'PGC4', 'PGC5', 'PGC6', 'PGC7', 'PGC8', 'chip']
df_gam3 = df.dropna(subset=[
    'prs_adhd_std', 'age_0_0', 'sex_plink', 'townsend',
    'cursmoke_0', 'curr_alcohol_0', 'adj_dbp_0'
] + genetic_covariates).copy()

#Recode categorical features
df_gam3['sex_model'] = df_gam3['sex_plink'] + 2
df_gam3['smoke_model'] = df_gam3['cursmoke_0'] + 2
df_gam3['alcohol_model'] = df_gam3['curr_alcohol_0'] + 2
df_gam3['chip_model'] = df_gam3['chip'].astype('category').cat.codes + 1  # For categorical encoding

#Prepare design matrix and response
X_gam3 = df_gam3[[
    'prs_adhd_std', 'age_0_0', 'sex_model',
    'townsend', 'smoke_model', 'alcohol_model',
    'PGC1', 'PGC2', 'PGC3', 'PGC4', 'PGC5', 'PGC6', 'PGC7', 'PGC8', 'chip_model'
]].values

y_gam3 = df_gam3['adj_dbp_0'].values

#Fit GAM model
gam3 = LinearGAM(
    s(0) + s(1) + f(2) + s(3) + f(4) + f(5) +  # PRS, age, sex, townsend, smoke, alcohol
    s(6) + s(7) + s(8) + s(9) + s(10) + s(11) + s(12) + s(13) +  # PGC1–PGC8
    f(14)  # chip
).fit(X_gam3, y_gam3)

#Build grid to visualize PRS effect only (term 0)
prs_vals = np.linspace(df_gam3['prs_adhd_std'].min(), df_gam3['prs_adhd_std'].max(), 200)
age_mean = df_gam3['age_0_0'].mean()
sex_mode = df_gam3['sex_model'].mode()[0]
townsend_mean = df_gam3['townsend'].mean()
smoke_mode = df_gam3['smoke_model'].mode()[0]
alcohol_mode = df_gam3['alcohol_model'].mode()[0]
pgc_means = [df_gam3[pgc].mean() for pgc in ['PGC1', 'PGC2', 'PGC3', 'PGC4', 'PGC5', 'PGC6', 'PGC7', 'PGC8']]
chip_mode = df_gam3['chip_model'].mode()[0]

X_grid = np.column_stack([
    prs_vals,
    np.full_like(prs_vals, age_mean),
    np.full_like(prs_vals, sex_mode),
    np.full_like(prs_vals, townsend_mean),
    np.full_like(prs_vals, smoke_mode),
    np.full_like(prs_vals, alcohol_mode),
    *[np.full_like(prs_vals, mean) for mean in pgc_means],
    np.full_like(prs_vals, chip_mode)
])

#Compute partial dependence
pdep, confi = gam3.partial_dependence(term=0, X=X_grid, width=0.95)

#Plot
plt.figure(figsize=(10, 6))
plt.plot(prs_vals, pdep, color='red', label='GAM Fit')
plt.fill_between(prs_vals, confi[:, 0], confi[:, 1], color='blue', alpha=0.3, label='95% CI')
plt.title("GAM 3: Effect of ADHD PRS on Adjusted DBP\n(Adjusted for Age, Sex, Lifestyle, and Genetic Covariates)")
plt.xlabel("Standardized ADHD PRS")
plt.ylabel("Partial Effect on DBP")
plt.legend()
plt.tight_layout()
plt.show()

#Summary
gam3.summary()


In [ ]:
#GAM 4 DBP ~ GAM 3 + BMI

#Clean and prepare data
genetic_covariates = ['PGC1', 'PGC2', 'PGC3', 'PGC4', 'PGC5', 'PGC6', 'PGC7', 'PGC8', 'chip']
df_gam4 = df.dropna(subset=[
    'prs_adhd_std', 'age_0_0', 'sex_plink', 'townsend',
    'cursmoke_0', 'curr_alcohol_0', 'adj_dbp_0', 'n_21001_0_0'
] + genetic_covariates).copy()

#Recode categorical variables for pyGAM (0/1 → 1/2)
df_gam4['sex_model'] = df_gam4['sex_plink'] + 1
df_gam4['smoke_model'] = df_gam4['cursmoke_0'] + 1
df_gam4['alcohol_model'] = df_gam4['curr_alcohol_0'] + 1

#Define predictors and response
X_gam4 = df_gam4[[
    'prs_adhd_std',     # 0
    'age_0_0',          # 1
    'sex_model',        # 2
    'townsend',         # 3
    'smoke_model',      # 4
    'alcohol_model',    # 5
    'n_21001_0_0',          # 6
] + genetic_covariates].values                    # 7–15 (PGC1–8, chip)

y_gam4 = df_gam4['adj_dbp_0'].values

#Fit the GAM model
gam4 = LinearGAM(
    s(0) +         # PRS
    s(1) +         # Age
    f(2) +         # Sex
    s(3) +         # Townsend
    f(4) +         # Smoking
    f(5) +         # Alcohol
    s(6) +         # BMI
    s(7) + s(8) + s(9) + s(10) + s(11) + s(12) + s(13) + s(14) +  # PCs
    f(15)          # chip
).fit(X_gam4, y_gam4)

#Plot PRS effect only
prs_vals = np.linspace(df_gam4['prs_adhd_std'].min(), df_gam4['prs_adhd_std'].max(), 200)
age_mean = df_gam4['age_0_0'].mean()
townsend_mean = df_gam4['townsend'].mean()
bmi_mean = df_gam4['n_21001_0_0'].mean()
sex_mode = df_gam4['sex_model'].mode()[0]
smoke_mode = df_gam4['smoke_model'].mode()[0]
alcohol_mode = df_gam4['alcohol_model'].mode()[0]
pc_means = df_gam4[genetic_covariates].mean().values

#Construct full grid for PRS effect
X_grid = np.column_stack([
    prs_vals,
    np.full_like(prs_vals, age_mean),
    np.full_like(prs_vals, sex_mode),
    np.full_like(prs_vals, townsend_mean),
    np.full_like(prs_vals, smoke_mode),
    np.full_like(prs_vals, alcohol_mode),
    np.full_like(prs_vals, bmi_mean),
    np.tile(pc_means, (len(prs_vals), 1))
])

#Partial dependence for PRS
pdep, confi = gam4.partial_dependence(term=0, X=X_grid, width=0.95)

#Plot
plt.figure(figsize=(10, 6))
plt.plot(prs_vals, pdep, color='red', label='GAM Fit')
plt.fill_between(prs_vals, confi[:, 0], confi[:, 1], color='blue', alpha=0.3, label='95% CI')
plt.title("GAM 4: Effect of ADHD PRS on Adjusted DBP\n(Adjusted for Age, Sex, Lifestyle, BMI, and Genetic Covariates)")
plt.xlabel("Standardized ADHD PRS")
plt.ylabel("Partial Effect on DBP")
plt.legend()
plt.tight_layout()
plt.show()

#summary
gam4.summary()


End of analysis of the relationship between PRS ADHD and phenotypes

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# COHORT DESCRIPTION TABLE
# ============================================================

# Create sex variable exactly as in your existing analysis
# sex_plink: 1 = male, 2 = female
df['sex'] = df['sex_plink'].map({1: 0, 2: 1})


# ------------------------------------------------------------
# 1. Check likely column names for variables not in this script
# ------------------------------------------------------------

print("Possible WHR / waist / hip columns:")
print([
    col for col in df.columns
    if any(x in col.lower() for x in ['whr', 'waist', 'hip'])
])

print("\nPossible HbA1c columns:")
print([
    col for col in df.columns
    if any(x in col.lower() for x in ['hba1c', 'hba', 'glycat'])
])

print("\nPossible cIMT columns:")
print([
    col for col in df.columns
    if any(x in col.lower() for x in ['cimt', 'imt', 'carotid'])
])


# ------------------------------------------------------------
# 2. Define variables whose names are confirmed
# ------------------------------------------------------------

continuous_vars = {
    'Age, years': 'age_0_0',
    'BMI, kg/m²': 'n_21001_0_0',
    'SBP, mmHg': 'adj_sbp_0',
    'DBP, mmHg': 'adj_dbp_0',
    'Townsend deprivation index': 'townsend'
}


# ------------------------------------------------------------
# 3. Continuous-variable summary
# ------------------------------------------------------------

continuous_rows = []

for label, col in continuous_vars.items():

    x = pd.to_numeric(df[col], errors='coerce').dropna()

    continuous_rows.append({
        'Characteristic': label,
        'N': len(x),
        'Mean': x.mean(),
        'SD': x.std(),
        'Mean ± SD': f"{x.mean():.2f} ± {x.std():.2f}"
    })

continuous_table = pd.DataFrame(continuous_rows)

print("\n==============================================")
print("CONTINUOUS CHARACTERISTICS")
print("==============================================")
print(continuous_table.to_string(index=False))


# ------------------------------------------------------------
# 4. Sex
# ------------------------------------------------------------

sex_valid = df['sex'].dropna()

n_total_sex = len(sex_valid)

n_male = (sex_valid == 0).sum()
n_female = (sex_valid == 1).sum()

male_pct = (n_male / n_total_sex) * 100
female_pct = (n_female / n_total_sex) * 100

print("\n==============================================")
print("SEX")
print("==============================================")
print(f"N with sex data: {n_total_sex:,}")
print(f"Male:   {n_male:,} ({male_pct:.2f}%)")
print(f"Female: {n_female:,} ({female_pct:.2f}%)")


# ------------------------------------------------------------
# 5. Current smoking
# ------------------------------------------------------------

smoking = df['cursmoke_0'].dropna()

print("\n==============================================")
print("CURRENT SMOKING")
print("==============================================")
print(smoking.value_counts(dropna=False))
print("\nPercentages:")
print((smoking.value_counts(normalize=True) * 100).round(2))


# ------------------------------------------------------------
# 6. Current alcohol use
# ------------------------------------------------------------

alcohol = df['curr_alcohol_0'].dropna()

print("\n==============================================")
print("CURRENT ALCOHOL USE")
print("==============================================")
print(alcohol.value_counts(dropna=False))
print("\nPercentages:")
print((alcohol.value_counts(normalize=True) * 100).round(2))


# ------------------------------------------------------------
# 7. Overall source dataset size
# ------------------------------------------------------------

print("\n==============================================")
print("OVERALL DATASET")
print("==============================================")
print(f"Total rows in source dataset: {len(df):,}")
print(f"Participants with PRS-ADHD: {df['prs_adhd_std'].notna().sum():,}")

In [ ]:
# WHR
x = pd.to_numeric(df["whr_0"], errors="coerce").dropna()

print("WHR N:", len(x))
print("WHR mean:", x.mean())
print("WHR SD:", x.std())


# Smoking
print("\nSmoking coding:")
print(df["cursmoke_0"].value_counts(dropna=False))
print("\nSmoking percentages:")
print((df["cursmoke_0"].value_counts(normalize=True, dropna=False) * 100).round(2))


# Alcohol
print("\nAlcohol coding:")
print(df["curr_alcohol_0"].value_counts(dropna=False))
print("\nAlcohol percentages:")
print((df["curr_alcohol_0"].value_counts(normalize=True, dropna=False) * 100).round(2))